# Tema 3 · Laboratorio — Una LSTM que recuerda

**Aprendizaje Profundo · CUNEF Universidad**

En este laboratorio entrenamos una **LSTM** en Keras para **predecir una serie temporal**: dada una ventana de valores pasados, adivinar el siguiente. Es la 'cinta de memoria' del Tema 3 en acción.

Al final la comparamos con un **MLP** (sin recurrencia) para ver qué aporta la memoria.

> Ejecuta las celdas en orden. En Colab no necesitas instalar nada.

## 1 · Una serie temporal

Creamos una señal sintética con **estacionalidad** (dos ondas), una **tendencia** ligera y algo de **ruido**. Suficiente para que la memoria del pasado ayude a predecir el futuro.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

T = 1200
t = np.arange(T)
serie = (np.sin(t * 2 * np.pi / 50)          # estacionalidad corta
         + 0.5 * np.sin(t * 2 * np.pi / 170)  # estacionalidad larga
         + 0.002 * t                          # tendencia
         + 0.15 * np.random.randn(T))         # ruido

plt.figure(figsize=(11, 3.5))
plt.plot(serie, lw=1); plt.title('Serie temporal sintética'); plt.grid(alpha=0.3); plt.show()

## 2 · Ventanas deslizantes

Una RNN espera secuencias. Convertimos la serie en pares `(ventana de N pasos) -> siguiente valor`. También normalizamos usando **solo** el tramo de entrenamiento (nada de mirar el futuro).

In [ ]:
N = 40  # tamaño de la ventana (cuántos pasos pasados ve la red)

# partición temporal: primeros 80 % train, resto test (¡sin barajar!)
split = int(T * 0.8)
mu, sd = serie[:split].mean(), serie[:split].std()
s = (serie - mu) / sd

def ventanas(arr, n):
    X, y = [], []
    for i in range(len(arr) - n):
        X.append(arr[i:i+n]); y.append(arr[i+n])
    return np.array(X), np.array(y)

X, y = ventanas(s, N)
# el índice i de X corresponde al valor original i+N
train_mask = (np.arange(len(X)) + N) < split
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]
# forma para RNN: (muestras, pasos, features)
X_train_r = X_train[..., None]
X_test_r = X_test[..., None]
print('train:', X_train_r.shape, ' test:', X_test_r.shape)

## 3 · La LSTM

Una capa `LSTM` que recorre la ventana paso a paso manteniendo su estado (la 'cinta de memoria'), seguida de una `Dense` que produce la predicción del siguiente valor.

In [ ]:
lstm = keras.Sequential([
    keras.layers.Input(shape=(N, 1)),
    layers.LSTM(32),
    layers.Dense(1),
])
lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm.summary()

hist = lstm.fit(X_train_r, y_train, validation_split=0.1,
                epochs=20, batch_size=32, verbose=2)

## 4 · Predicción vs. realidad

Predecimos el tramo de test (que la red no vio) y superponemos la predicción sobre el valor real.

In [ ]:
pred = lstm.predict(X_test_r, verbose=0).ravel()

plt.figure(figsize=(11, 4))
plt.plot(y_test, label='real', lw=1.6)
plt.plot(pred, label='predicción LSTM', lw=1.4)
plt.title('LSTM · predicción en el tramo de test'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

mae_lstm = np.mean(np.abs(pred - y_test))
print(f'MAE de la LSTM en test: {mae_lstm:.4f}')

## 5 · ¿Y sin memoria? Un MLP

El MLP ve exactamente la misma ventana de 40 valores, pero **aplanada**: no la recorre como una secuencia, no tiene estado. Comparamos su error con el de la LSTM.

In [ ]:
mlp = keras.Sequential([
    keras.layers.Input(shape=(N,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])
mlp.compile(optimizer='adam', loss='mse', metrics=['mae'])
mlp.fit(X_train, y_train, validation_split=0.1, epochs=20, batch_size=32, verbose=0)

pred_mlp = mlp.predict(X_test, verbose=0).ravel()
mae_mlp = np.mean(np.abs(pred_mlp - y_test))
print(f'MAE de la LSTM: {mae_lstm:.4f}')
print(f'MAE del MLP   : {mae_mlp:.4f}')
print('\nNota: en esta serie sencilla los dos aciertan bastante; la ventaja de la')
print('LSTM crece cuando las dependencias temporales son más largas y complejas.')

## 6 · Tus retos

1. **La ventana.** Baja `N` a 5 y súbelo a 100. ¿Cómo cambia el error? ¿Ayuda ver más pasado?
2. **Más memoria.** Prueba `LSTM(64)` o apila dos capas (`LSTM(32, return_sequences=True)` + `LSTM(32)`). ¿Mejora?
3. **Serie más difícil.** Sube el ruido a `0.4` o alarga las estacionalidades. ¿Se abre la brecha entre LSTM y MLP?
4. **Texto (avanzado).** Adapta la idea a una secuencia de caracteres: predecir el siguiente carácter de un texto con una LSTM y `Embedding`.

Cuando termines, vuelve a la [práctica interactiva](../../practica-t3.html) y relaciona el estado de la LSTM con la 'cinta de memoria' que manipulabas allí.